## Libraries and 

In [1]:
# Import Libraries

import pandas as pd
import numpy as np

from pathlib import Path

import joblib


In [2]:
# Set File Paths

DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")

##### Load Saved Models and Files

In [3]:
# Load Saved Models and Files

# Supervised model
prediction_model = joblib.load(MODEL_DIR / "energy_consumption_prediction_model.pkl")
model_features = joblib.load(MODEL_DIR / "energy_model_features.pkl")
category_thresholds = joblib.load(MODEL_DIR / "consumption_category_thresholds.pkl")

# Clustering model
kmeans_model = joblib.load(MODEL_DIR / "energy_kmeans_model.pkl")
cluster_scaler = joblib.load(MODEL_DIR / "energy_cluster_scaler.pkl")
clustering_features = joblib.load(MODEL_DIR / "energy_clustering_features.pkl")
cluster_segment_mapping = joblib.load(MODEL_DIR / "cluster_segment_mapping.pkl")

# Anomaly detection model
anomaly_model = joblib.load(MODEL_DIR / "energy_anomaly_model.pkl")
anomaly_scaler = joblib.load(MODEL_DIR / "energy_anomaly_scaler.pkl")
anomaly_features = joblib.load(MODEL_DIR / "energy_anomaly_features.pkl")
anomaly_thresholds = joblib.load(MODEL_DIR / "anomaly_thresholds.pkl")

print("All saved models and files loaded successfully!")

All saved models and files loaded successfully!


In [4]:
# Load Final Dataset

energy_df = pd.read_csv(DATA_DIR / "processed" / "energy_household_with_anomalies.csv")

print("Dataset loaded successfully!")
print("Shape:", energy_df.shape)

energy_df.head()

Dataset loaded successfully!
Shape: (3599, 66)


,hhid,zone,state,lga,urca_cat,adult_count,members_under_18,members_18_30,members_31_55,members_over_55,...,average_ac_usage_hours,kmeans_cluster,hierarchical_cluster,energy_segment,pca_1,pca_2,anomaly_label,anomaly_score,anomaly_status,anomaly_warning
0,d8af8ab5-30ab-4d9f-bfe4-81231dbe5dbf,North Central,Niger,mashegu,<1hr to small city/town+,5,13,4,1,0,...,0,1,0,Moderate Fan-Heavy Household,2.595801,3.470226,1.0,0.032675,Normal,Normal Usage
1,e8245d5c-8130-4e78-b4b0-1053b7ecbc9b,North Central,Niger,mashegu,<1hr to small city/town+,6,4,3,2,1,...,0,1,0,Low-Use Basic Household,-0.172048,1.651435,1.0,0.166829,Normal,Normal Usage
2,435c8e27-517a-46b9-af04-48830e086d7a,North West,Kano,garun_malam,<1hr to large city,3,3,2,1,0,...,0,1,0,High Consumption Household,-0.117451,-0.030571,1.0,0.189698,Normal,Normal Usage
3,9303fa53-9fd2-41a9-9f0d-9567dbe5168e,North West,Kano,garun_malam,<1hr to large city,3,7,2,1,0,...,0,1,0,Appliance-Heavy High Consumption,3.432582,2.378012,1.0,0.038461,Normal,Normal Usage
4,c62cc5a5-29c5-423b-9543-a7b05bda454b,North West,Kano,garun_malam,<1hr to large city,2,4,1,1,0,...,0,1,0,Low-Use Basic Household,-1.875530,-0.698028,1.0,0.213147,Normal,Normal Usage


## Service Band to Supply Hours Function

In [23]:
# Service Band to Supply Hours
# The user will select a service band. The app will automatically convert it to daily supply hours

def band_to_supply_hours(service_band):
    band_hours = {
        "Band A": 20,
        "Band B": 16,
        "Band C": 12,
        "Band D": 8,
        "Band E": 4,
        "Below Band E / Low Supply": 2
    }

    return band_hours.get(service_band, 12)

## Create User Input Profile Function

In [6]:
# Create User Input Profile

def create_user_profile(
    household_size,
    number_of_rooms,
    service_band,
    light_bulb_count,
    fan_count,
    television_count,
    fridge_count,
    ac_count,
    average_ac_usage_hours
):
    daily_supply_hours = band_to_supply_hours(service_band)

    user_profile = {
        "household_size": household_size,
        "number_of_rooms": number_of_rooms,
        "daily_supply_hours": daily_supply_hours,
        "light_bulb_count": light_bulb_count,
        "fan_count": fan_count,
        "television_count": television_count,
        "fridge_count": fridge_count,
        "ac_count": ac_count,
        "average_ac_usage_hours": average_ac_usage_hours
    }

    return user_profile

## Predict Monthly Electricity Consumption

In [7]:
# Predict Monthly Electricity Consumption

def predict_monthly_consumption(user_profile):
    input_df = pd.DataFrame([user_profile])

    input_df = input_df[model_features]

    predicted_kwh = prediction_model.predict(input_df)[0]

    predicted_kwh = max(predicted_kwh, 0)

    return predicted_kwh

## Categorize Predicted Consumption

In [8]:
# Categorize Predicted Consumption

def categorize_consumption(kwh):
    low_threshold = category_thresholds["low_threshold"]
    moderate_threshold = category_thresholds["moderate_threshold"]
    high_threshold = category_thresholds["high_threshold"]

    if kwh <= low_threshold:
        return "Low Consumption"
    elif kwh <= moderate_threshold:
        return "Moderate Consumption"
    elif kwh <= high_threshold:
        return "High Consumption"
    else:
        return "Very High Consumption"

## Estimate Monthly Cost

In [9]:
# Estimate Monthly Cost

def estimate_monthly_cost(predicted_kwh, tariff_per_kwh):
    estimated_cost = predicted_kwh * tariff_per_kwh

    return estimated_cost

## Assign Household Energy Segment

In [10]:
# Assign Household Energy Segment

def assign_energy_segment(user_profile, predicted_kwh):
    cluster_input = user_profile.copy()

    cluster_input["estimated_monthly_kwh"] = predicted_kwh

    cluster_df = pd.DataFrame([cluster_input])

    cluster_df = cluster_df[clustering_features]

    cluster_scaled = cluster_scaler.transform(cluster_df)

    cluster_number = int(kmeans_model.predict(cluster_scaled)[0])

    clean_mapping = {
        int(key): value for key, value in cluster_segment_mapping.items()
    }

    energy_segment = clean_mapping.get(
        cluster_number,
        "General Household Energy Segment"
    )

    return cluster_number, energy_segment

## Detect Anomaly

In [11]:
# Detect Anomaly

def detect_anomaly(user_profile, predicted_kwh):
    anomaly_input = user_profile.copy()

    anomaly_input["estimated_monthly_kwh"] = predicted_kwh

    anomaly_df = pd.DataFrame([anomaly_input])

    anomaly_df = anomaly_df[anomaly_features]

    anomaly_scaled = anomaly_scaler.transform(anomaly_df)

    anomaly_label = int(anomaly_model.predict(anomaly_scaled)[0])

    anomaly_score = anomaly_model.decision_function(anomaly_scaled)[0]

    high_consumption_threshold = anomaly_thresholds["high_consumption_threshold"]

    if anomaly_label == -1 and predicted_kwh >= high_consumption_threshold:
        anomaly_warning = "High Consumption Anomaly"
    elif anomaly_label == -1:
        anomaly_warning = "Unusual Low Consumption"
    else:
        anomaly_warning = "Normal Usage"

    return anomaly_warning, anomaly_score

## Create Appliance Usage Breakdown

In [12]:
# Create Appliance Usage Breakdown

def calculate_appliance_breakdown(user_profile):
    daily_supply_hours = user_profile["daily_supply_hours"]

    appliance_data = [
        {
            "Appliance": "Lighting",
            "Count": user_profile["light_bulb_count"],
            "Wattage": 10,
            "Daily Hours": min(daily_supply_hours, 6)
        },
        {
            "Appliance": "Fans",
            "Count": user_profile["fan_count"],
            "Wattage": 60,
            "Daily Hours": min(daily_supply_hours, 8)
        },
        {
            "Appliance": "Television",
            "Count": user_profile["television_count"],
            "Wattage": 100,
            "Daily Hours": min(daily_supply_hours, 4)
        },
        {
            "Appliance": "Fridge / Freezer",
            "Count": user_profile["fridge_count"],
            "Wattage": 150,
            "Daily Hours": 12 if user_profile["fridge_count"] > 0 else 0
        },
        {
            "Appliance": "Air Conditioner",
            "Count": user_profile["ac_count"],
            "Wattage": 1000,
            "Daily Hours": user_profile["average_ac_usage_hours"]
        }
    ]

    breakdown_df = pd.DataFrame(appliance_data)

    breakdown_df["Estimated Monthly kWh"] = (
        breakdown_df["Count"] *
        breakdown_df["Wattage"] *
        breakdown_df["Daily Hours"] *
        30 / 1000
    )

    breakdown_df = breakdown_df.sort_values(
        by="Estimated Monthly kWh",
        ascending=False
    )

    return breakdown_df

## Generate Energy-Saving Recommendations

In [13]:
# Generate Energy-Saving Recommendations

def generate_recommendations(
    user_profile,
    predicted_kwh,
    consumption_category,
    energy_segment,
    anomaly_warning,
    tariff_per_kwh,
    max_recommendations=3
):
    recommendations = []

    def add_recommendation(focus_area, message, monthly_kwh_saving=0):
        monthly_cost_saving = monthly_kwh_saving * tariff_per_kwh

        recommendations.append({
            "Focus Area": focus_area,
            "Recommendation": message,
            "Potential Monthly Saving": f"{round(monthly_kwh_saving, 2)} kWh / ₦{round(monthly_cost_saving, 2)}"
        })

    breakdown_df = calculate_appliance_breakdown(user_profile)

    top_appliance = breakdown_df.iloc[0]["Appliance"]

    if anomaly_warning == "High Consumption Anomaly":
        add_recommendation(
            "Usage Check",
            "Your usage looks unusually high for this household profile. Review high-energy appliances and check for waste or inefficient usage.",
            0
        )

    if user_profile["ac_count"] > 0 and user_profile["average_ac_usage_hours"] >= 3:
        saving_kwh = user_profile["ac_count"] * 1000 * 1 * 30 / 1000

        add_recommendation(
            "Air Conditioner",
            "Reduce AC usage by at least 1 hour daily or use fan support when possible.",
            saving_kwh
        )

    if user_profile["light_bulb_count"] >= 6:
        bulbs_to_reduce = min(user_profile["light_bulb_count"], 5)

        saving_kwh = bulbs_to_reduce * 10 * 2 * 30 / 1000

        add_recommendation(
            "Lighting",
            "Switch off unused bulbs and use LED bulbs where possible.",
            saving_kwh
        )

    if user_profile["fan_count"] >= 3:
        saving_kwh = user_profile["fan_count"] * 60 * 1 * 30 / 1000

        add_recommendation(
            "Fans",
            "Turn off fans in empty rooms and use natural ventilation when available.",
            saving_kwh
        )

    if user_profile["fridge_count"] > 0:
        add_recommendation(
            "Fridge / Freezer",
            "Avoid frequent opening, check door seals, and do not overload the fridge.",
            0
        )

    if consumption_category in ["High Consumption", "Very High Consumption"]:
        add_recommendation(
            "Energy Monitoring",
            "Track your high-use appliances weekly and reduce unnecessary usage during peak hours.",
            0
        )

    if user_profile["daily_supply_hours"] <= 8:
        add_recommendation(
            "Limited Supply",
            "Prioritize essential appliances during limited supply hours to reduce waste.",
            0
        )

    if len(recommendations) == 0:
        add_recommendation(
            "Good Usage Pattern",
            "Your household energy profile looks reasonable. Continue switching off appliances when not in use.",
            0
        )

    recommendation_df = pd.DataFrame(recommendations)

    recommendation_df = recommendation_df.drop_duplicates(
        subset=["Focus Area"],
        keep="first"
    )

    recommendation_df = recommendation_df.head(max_recommendations)

    return recommendation_df, breakdown_df

## Create Simple Anomaly Message

In [14]:
# Create Simple Anomaly Message

def get_anomaly_message(anomaly_warning):
    if anomaly_warning == "High Consumption Anomaly":
        return "This household appears to have unusually high electricity consumption for its profile."

    elif anomaly_warning == "Unusual Low Consumption":
        return "This household has unusually low estimated consumption. This may be due to limited electricity supply or very few appliances."

    else:
        return "No unusual electricity usage pattern was detected."

## Run Full Recommendation System

In [15]:
# Run Full Recommendation System

def run_recommendation_system(
    household_size,
    number_of_rooms,
    service_band,
    light_bulb_count,
    fan_count,
    television_count,
    fridge_count,
    ac_count,
    average_ac_usage_hours,
    tariff_per_kwh
):
    user_profile = create_user_profile(
        household_size=household_size,
        number_of_rooms=number_of_rooms,
        service_band=service_band,
        light_bulb_count=light_bulb_count,
        fan_count=fan_count,
        television_count=television_count,
        fridge_count=fridge_count,
        ac_count=ac_count,
        average_ac_usage_hours=average_ac_usage_hours
    )

    predicted_kwh = predict_monthly_consumption(user_profile)

    estimated_cost = estimate_monthly_cost(
        predicted_kwh,
        tariff_per_kwh
    )

    consumption_category = categorize_consumption(predicted_kwh)

    cluster_number, energy_segment = assign_energy_segment(
        user_profile,
        predicted_kwh
    )

    anomaly_warning, anomaly_score = detect_anomaly(
        user_profile,
        predicted_kwh
    )

    anomaly_message = get_anomaly_message(anomaly_warning)

    recommendation_df, breakdown_df = generate_recommendations(
        user_profile=user_profile,
        predicted_kwh=predicted_kwh,
        consumption_category=consumption_category,
        energy_segment=energy_segment,
        anomaly_warning=anomaly_warning,
        tariff_per_kwh=tariff_per_kwh,
        max_recommendations=3
    )

    summary = {
        "Predicted Monthly Consumption": round(predicted_kwh, 2),
        "Estimated Monthly Cost": round(estimated_cost, 2),
        "Consumption Category": consumption_category,
        "Energy Segment": energy_segment,
        "Anomaly Status": anomaly_warning,
        "Anomaly Message": anomaly_message
    }

    return summary, recommendation_df, breakdown_df

## TeamRed Testing With Sample Household Data

In [16]:
# Test With a Sample Household

summary, recommendation_df, breakdown_df = run_recommendation_system(
    household_size=4,
    number_of_rooms=3,
    service_band="Band A",
    light_bulb_count=8,
    fan_count=3,
    television_count=1,
    fridge_count=1,
    ac_count=1,
    average_ac_usage_hours=4,
    tariff_per_kwh=225
)

print("Household Energy Summary")
print("------------------------")

for key, value in summary.items():
    if key == "Estimated Monthly Cost":
        print(f"{key}: ₦{value:,.2f}")
    elif key == "Predicted Monthly Consumption":
        print(f"{key}: {value} kWh")
    else:
        print(f"{key}: {value}")

Household Energy Summary
------------------------
Predicted Monthly Consumption: 234.3 kWh
Estimated Monthly Cost: ₦52,717.31
Consumption Category: Very High Consumption
Energy Segment: Cooling-Heavy High Consumption
Anomaly Status: High Consumption Anomaly
Anomaly Message: This household appears to have unusually high electricity consumption for its profile.


In [17]:
# View Recommendation Output

recommendation_df

,Focus Area,Recommendation,Potential Monthly Saving
0,Usage Check,Your usage looks unusually high for this house...,0 kWh / ₦0
1,Air Conditioner,Reduce AC usage by at least 1 hour daily or us...,30.0 kWh / ₦6750.0
2,Lighting,Switch off unused bulbs and use LED bulbs wher...,3.0 kWh / ₦675.0


In [18]:
# View Appliance Breakdown

breakdown_df

,Appliance,Count,Wattage,Daily Hours,Estimated Monthly kWh
4,Air Conditioner,1,1000,4,120.0
3,Fridge / Freezer,1,150,12,54.0
1,Fans,3,60,8,43.2
0,Lighting,8,10,6,14.4
2,Television,1,100,4,12.0


## TeamRed Testing

In [19]:
# Test a Low-Use Household

low_summary, low_recommendation_df, low_breakdown_df = run_recommendation_system(
    household_size=2,
    number_of_rooms=1,
    service_band="Band E",
    light_bulb_count=2,
    fan_count=1,
    television_count=0,
    fridge_count=0,
    ac_count=0,
    average_ac_usage_hours=0,
    tariff_per_kwh=100
)

print("Low-Use Household Summary")
print("-------------------------")

for key, value in low_summary.items():
    if key == "Estimated Monthly Cost":
        print(f"{key}: ₦{value:,.2f}")
    elif key == "Predicted Monthly Consumption":
        print(f"{key}: {value} kWh")
    else:
        print(f"{key}: {value}")

low_recommendation_df

Low-Use Household Summary
-------------------------
Predicted Monthly Consumption: 1.46 kWh
Estimated Monthly Cost: ₦145.87
Consumption Category: Low Consumption
Energy Segment: Low-Use Basic Household
Anomaly Status: Normal Usage
Anomaly Message: No unusual electricity usage pattern was detected.


,Focus Area,Recommendation,Potential Monthly Saving
0,Limited Supply,Prioritize essential appliances during limited...,0 kWh / ₦0


In [20]:
# Test a High-Use Household

high_summary, high_recommendation_df, high_breakdown_df = run_recommendation_system(
    household_size=6,
    number_of_rooms=5,
    service_band="Band A",
    light_bulb_count=15,
    fan_count=6,
    television_count=3,
    fridge_count=2,
    ac_count=2,
    average_ac_usage_hours=6,
    tariff_per_kwh=225
)

print("High-Use Household Summary")
print("--------------------------")

for key, value in high_summary.items():
    if key == "Estimated Monthly Cost":
        print(f"{key}: ₦{value:,.2f}")
    elif key == "Predicted Monthly Consumption":
        print(f"{key}: {value} kWh")
    else:
        print(f"{key}: {value}")

high_recommendation_df

High-Use Household Summary
--------------------------
Predicted Monthly Consumption: 348.29 kWh
Estimated Monthly Cost: ₦78,365.97
Consumption Category: Very High Consumption
Energy Segment: Cooling-Heavy High Consumption
Anomaly Status: High Consumption Anomaly
Anomaly Message: This household appears to have unusually high electricity consumption for its profile.


,Focus Area,Recommendation,Potential Monthly Saving
0,Usage Check,Your usage looks unusually high for this house...,0 kWh / ₦0
1,Air Conditioner,Reduce AC usage by at least 1 hour daily or us...,60.0 kWh / ₦13500.0
2,Lighting,Switch off unused bulbs and use LED bulbs wher...,3.0 kWh / ₦675.0


## Save Sample Recommendation Outputs

In [21]:
# Save Sample Recommendation Outputs

recommendation_output_dir = DATA_DIR / "processed"

recommendation_df.to_csv(
    recommendation_output_dir / "sample_recommendations.csv",
    index=False
)

breakdown_df.to_csv(
    recommendation_output_dir / "sample_appliance_breakdown.csv",
    index=False
)

print("Sample recommendation outputs saved successfully!")

Sample recommendation outputs saved successfully!
